# AGENTS026 — GPU-Powered RCA Agent
**AMD Instinct MI300X · Qwen3-30B via vLLM · MiniCluster Banking Stack**

This notebook runs the full autonomous loop:
1. Read live telemetry from `live_metrics.csv`
2. Detect anomalies (threshold + statistical)
3. Call Qwen3-30B on GPU for Root Cause Analysis
4. Decide: auto-remediate or escalate to HITL
5. Execute remediation via fault injection APIs
6. Write results to audit log and HITL queue (visible in Streamlit console)

## Cell 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import json, time, uuid, requests
from datetime import datetime, timezone
from pathlib import Path
from openai import OpenAI
from IPython.display import display, HTML, clear_output

# ── paths ──────────────────────────────────────────────────────────────────
BASE          = Path('/workspace/shared')
METRICS_CSV   = BASE / 'minicluster' / 'live_metrics.csv'
HITL_FILE     = BASE / 'hitl_queue.jsonl'
AUDIT_FILE    = BASE / 'audit_log.jsonl'
INCIDENTS_DIR = BASE / 'incidents'
INCIDENTS_DIR.mkdir(parents=True, exist_ok=True)
HITL_FILE.parent.mkdir(parents=True, exist_ok=True)
AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)

# ── vLLM client (GPU) ──────────────────────────────────────────────────────
llm = OpenAI(base_url='http://localhost:8000/v1', api_key='abc-123')
MODEL = 'Qwen3-30B-A3B'

# ── service ports ──────────────────────────────────────────────────────────
SERVICES = {'payments': 7001, 'auth': 7002, 'checkout': 7003, 'fraud': 7004}

# ── thresholds ─────────────────────────────────────────────────────────────
THRESHOLDS = {
    'cpu_utilization': 70.0,
    'latency_p95_ms':  500.0,
    'error_rate':      0.05,
    'mem_mb':          800.0,
}

# ── remediation confidence gate ────────────────────────────────────────────
# confidence >= this → auto-remediate; below → HITL
AUTO_REMEDIATE_THRESHOLD = 0.75

print('✅ Config loaded')
print(f'   Metrics CSV : {METRICS_CSV}')
print(f'   HITL file   : {HITL_FILE}')
print(f'   Audit file  : {AUDIT_FILE}')
print(f'   LLM model   : {MODEL}')

## Cell 2 — Verify GPU / vLLM is alive

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['rocm-smi', '--showuse'], capture_output=True, text=True)
print('=== GPU Status ===')
print(result.stdout or result.stderr)

# Check vLLM
try:
    models = llm.models.list()
    print(f'\n✅ vLLM alive — models: {[m.id for m in models.data]}')
except Exception as e:
    print(f'❌ vLLM not reachable: {e}')
    print('   Make sure Terminal 1 vLLM is running and fully loaded')

## Cell 3 — Helper Functions

In [ ]:
def ts():
    return datetime.now(timezone.utc).isoformat()

def load_metrics(window_mins=10):
    """Load recent metrics from live_metrics.csv"""
    df = pd.read_csv(METRICS_CSV, parse_dates=['timestamp'])
    cutoff = df['timestamp'].max() - pd.Timedelta(minutes=window_mins)
    return df[df['timestamp'] >= cutoff].copy()

def detect_anomalies(df):
    """Threshold + z-score anomaly detection"""
    anomalies = []
    latest = df.sort_values('timestamp').groupby('service').last().reset_index()

    for _, row in latest.iterrows():
        svc = row['service']
        for metric, thresh in THRESHOLDS.items():
            if metric not in row:
                continue
            val = float(row[metric])
            if val > thresh:
                # z-score over recent window for severity
                svc_df = df[df['service'] == svc][metric].dropna()
                z = (val - svc_df.mean()) / (svc_df.std() + 1e-9)
                severity = 'CRITICAL' if z > 3 else ('HIGH' if val > thresh * 1.5 else 'WARN')
                anomalies.append({
                    'service':   svc,
                    'metric':    metric,
                    'value':     round(val, 4),
                    'threshold': thresh,
                    'z_score':   round(float(z), 2),
                    'severity':  severity,
                    'node':      row.get('node', 'unknown'),
                    'timestamp': str(row['timestamp']),
                })
    return anomalies

def fault_post(port, path, payload):
    """Call fault injection API"""
    try:
        r = requests.post(f'http://127.0.0.1:{port}{path}', json=payload, timeout=3)
        return r.json()
    except Exception as e:
        return {'error': str(e)}

def write_hitl(event):
    with open(HITL_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def write_audit(event):
    with open(AUDIT_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def save_incident(incident):
    path = INCIDENTS_DIR / f"{incident['incident_id']}.json"
    with open(path, 'w') as f:
        json.dump(incident, f, indent=2, default=str)

print('✅ Helper functions loaded')

## Cell 4 — GPU RCA Agent (core LLM call)

In [ ]:
def run_rca_agent(anomalies, metrics_summary):
    """
    Call Qwen3-30B on GPU for root cause analysis.
    Returns structured JSON with: root_cause, confidence, action, auto_remediate
    """
    system_prompt = """You are an autonomous SRE agent for a banking microservices platform.
You diagnose incidents and decide remediation actions.

Services: payments (7001), auth (7002), checkout (7003), fraud (7004)
All run on AMD MI300X GPU cluster.

Fault injection endpoints available:
- POST /fault/latency  {"ms": N}         — adds latency
- POST /fault/errors   {"pct": 0.0-1.0}  — injects error rate  
- POST /fault/cpu_spin {"seconds": N}    — CPU spike
- POST /fault/mem_leak {"mb_per_min": N} — memory leak
- POST /fault/clear    {}                — clears ALL faults on a service

Remediation actions you can take:
- clear_fault: clear all faults on affected service (safe, always auto-approve)
- restart_service: restart the service (medium risk, auto-approve if confidence > 0.8)
- scale_service: scale out (low risk)
- rollback_config: rollback config change (high risk, always send to HITL)
- drain_node: drain the node (high risk, always send to HITL)
- escalate: cannot determine cause, human needed

Respond ONLY with valid JSON, no markdown, no explanation outside JSON:
{
  "root_cause": "one sentence explanation",
  "root_cause_detail": "2-3 sentence technical detail",
  "affected_services": ["service1"],
  "confidence": 0.0-1.0,
  "action": "clear_fault|restart_service|scale_service|rollback_config|drain_node|escalate",
  "action_target": "service_name",
  "action_params": {},
  "auto_remediate": true|false,
  "reasoning": "why this action",
  "risk_level": "LOW|MEDIUM|HIGH"
}"""

    user_msg = f"""ANOMALIES DETECTED:
{json.dumps(anomalies, indent=2)}

RECENT METRICS SUMMARY:
{metrics_summary}

Diagnose the root cause and decide on remediation action."""

    print('🧠 Calling Qwen3-30B on GPU for RCA...')
    t0 = time.time()

    response = llm.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_msg}
        ],
        temperature=0.1,
        max_tokens=800,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}}
    )

    elapsed = time.time() - t0
    raw = response.choices[0].message.content.strip()
    tokens = response.usage.completion_tokens
    print(f'   ✅ LLM responded in {elapsed:.1f}s — {tokens} tokens ({tokens/elapsed:.0f} tok/s)')

    # parse JSON
    try:
        # strip any accidental markdown
        clean = raw.replace('```json','').replace('```','').strip()
        result = json.loads(clean)
    except json.JSONDecodeError:
        print(f'   ⚠️  JSON parse failed, raw: {raw[:200]}')
        result = {
            'root_cause': 'LLM response parse error',
            'confidence': 0.0,
            'action': 'escalate',
            'auto_remediate': False,
            'risk_level': 'HIGH',
            'raw_response': raw
        }

    result['llm_latency_s']  = round(elapsed, 2)
    result['tokens_used']    = tokens
    return result

print('✅ RCA agent function loaded')

## Cell 5 — Remediation Executor

In [ ]:
def execute_remediation(rca, incident_id):
    """
    Execute the RCA agent's recommended action.
    Returns execution result dict.
    """
    action  = rca.get('action', 'escalate')
    target  = rca.get('action_target', '')
    params  = rca.get('action_params', {})
    port    = SERVICES.get(target)

    result = {
        'incident_id': incident_id,
        'action':      action,
        'target':      target,
        'timestamp':   ts(),
        'status':      'UNKNOWN',
        'detail':      ''
    }

    if action == 'clear_fault' and port:
        r = fault_post(port, '/fault/clear', {})
        result['status'] = 'EXECUTED'
        result['detail'] = f'Cleared all faults on {target}: {r}'
        print(f'   🔧 Cleared faults on {target}: {r}')

    elif action == 'restart_service' and port:
        # Simulate restart via clear + brief delay
        r = fault_post(port, '/fault/clear', {})
        result['status'] = 'EXECUTED'
        result['detail'] = f'Simulated restart of {target} (clear + reset): {r}'
        print(f'   🔄 Restarted {target}: {r}')

    elif action == 'scale_service':
        result['status'] = 'SIMULATED'
        result['detail'] = f'Scale-out of {target} simulated (no real infra API)'
        print(f'   📈 Scale-out {target} simulated')

    elif action in ('rollback_config', 'drain_node', 'escalate'):
        result['status'] = 'SKIPPED'
        result['detail'] = f'High-risk action {action} — sent to HITL instead'
        print(f'   ⚠️  {action} is high-risk — routing to HITL')

    else:
        result['status'] = 'UNKNOWN_ACTION'
        result['detail'] = f'No executor for action: {action}'
        print(f'   ❓ Unknown action: {action}')

    return result

print('✅ Remediation executor loaded')

## Cell 6 — Full Autonomous Loop (single run)

In [ ]:
def run_autonomous_loop(verbose=True):
    """
    One full iteration:
    detect → RCA (GPU) → decide → execute or HITL → audit
    """
    incident_id = f'inc-{uuid.uuid4().hex[:8]}'
    loop_start  = time.time()

    print(f'\n{"="*60}')
    print(f'🔁 AUTONOMOUS LOOP — {incident_id}')
    print(f'   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'{"="*60}')

    # ── Step 1: load metrics ───────────────────────────────────────────────
    df = load_metrics(window_mins=10)
    if df.empty:
        print('⚠️  No metrics data yet — is the collector running?')
        return None

    latest = df.sort_values('timestamp').groupby('service').last()
    metrics_summary = latest[['cpu_utilization','latency_p95_ms',
                               'error_rate','mem_mb']].round(3).to_string()
    print(f'\n📊 Latest metrics:\n{metrics_summary}')

    # ── Step 2: detect anomalies ───────────────────────────────────────────
    anomalies = detect_anomalies(df)
    print(f'\n🔍 Anomalies detected: {len(anomalies)}')

    if not anomalies:
        print('   ✅ All services healthy — no action needed')
        write_audit({
            'event_type':   'LOOP_RUN_HEALTHY',
            'incident_id':  incident_id,
            'timestamp':    ts(),
            'anomaly_count': 0
        })
        return {'status': 'healthy', 'incident_id': incident_id}

    for a in anomalies:
        sev_icon = '🔴' if a['severity'] == 'CRITICAL' else ('🟡' if a['severity'] == 'HIGH' else '⚪')
        print(f'   {sev_icon} {a["service"]}.{a["metric"]} = {a["value"]} '
              f'(thresh={a["threshold"]}, z={a["z_score"]}, {a["severity"]})')

    # ── Step 3: GPU RCA ────────────────────────────────────────────────────
    print(f'\n🧠 Running GPU-based RCA...')
    rca = run_rca_agent(anomalies, metrics_summary)

    print(f'\n📋 RCA Result:')
    print(f'   Root cause  : {rca.get("root_cause")}')
    print(f'   Confidence  : {rca.get("confidence")}')
    print(f'   Action      : {rca.get("action")} → {rca.get("action_target")}')
    print(f'   Risk level  : {rca.get("risk_level")}')
    print(f'   Auto-fix    : {rca.get("auto_remediate")}')
    print(f'   Reasoning   : {rca.get("reasoning")}')

    # ── Step 4: decide — auto or HITL ─────────────────────────────────────
    confidence     = float(rca.get('confidence', 0))
    auto_remediate = rca.get('auto_remediate', False)
    risk_level     = rca.get('risk_level', 'HIGH')

    # Force HITL for high-risk actions regardless of confidence
    if risk_level == 'HIGH' or rca.get('action') in ('rollback_config','drain_node','escalate'):
        auto_remediate = False

    # Force HITL if confidence too low
    if confidence < AUTO_REMEDIATE_THRESHOLD:
        auto_remediate = False

    exec_result = None

    if auto_remediate:
        print(f'\n⚡ AUTO-REMEDIATE (confidence={confidence:.2f} >= {AUTO_REMEDIATE_THRESHOLD})')
        exec_result = execute_remediation(rca, incident_id)
        write_audit({
            'event_type':    'AUTO_REMEDIATION',
            'incident_id':   incident_id,
            'timestamp':     ts(),
            'anomalies':     anomalies,
            'rca':           rca,
            'exec_result':   exec_result,
        })
        print(f'   Status: {exec_result["status"]} — {exec_result["detail"]}')
    else:
        print(f'\n🛑 ESCALATING TO HITL '
              f'(confidence={confidence:.2f}, risk={risk_level}, auto={auto_remediate})')
        hitl_event = {
            'hitl_id':     f'hitl-{uuid.uuid4().hex[:8]}',
            'incident_id': incident_id,
            'timestamp':   ts(),
            'source':      'rca_agent',
            'anomalies':   anomalies,
            'rca':         rca,
            'status':      'PENDING',
            'operator':    None,
        }
        write_hitl(hitl_event)
        write_audit({**hitl_event, 'event_type': 'HITL_CREATED'})
        print(f'   ✅ Written to HITL queue — check Streamlit console HITL tab')

    # ── Step 5: save incident ──────────────────────────────────────────────
    incident = {
        'incident_id':   incident_id,
        'timestamp':     ts(),
        'anomalies':     anomalies,
        'rca':           rca,
        'exec_result':   exec_result,
        'auto_remediated': auto_remediate,
        'loop_duration_s': round(time.time() - loop_start, 2),
    }
    save_incident(incident)

    print(f'\n✅ Loop complete in {incident["loop_duration_s"]}s')
    print(f'   Incident saved: {INCIDENTS_DIR}/{incident_id}.json')
    return incident

print('✅ Autonomous loop function loaded')

## Cell 7 — Inject a Fault + Run Loop (Demo)

In [ ]:
# Step 1: Inject a real fault into payments service
print('💥 Injecting fault: 800ms latency + 20% errors on payments...')
r1 = requests.post('http://127.0.0.1:7001/fault/latency', json={'ms': 800}, timeout=3)
r2 = requests.post('http://127.0.0.1:7001/fault/errors',  json={'pct': 0.2}, timeout=3)
print(f'   Latency: {r1.json()}')
print(f'   Errors : {r2.json()}')

print('\n⏳ Waiting 70s for metrics to capture the fault...')
for i in range(70, 0, -10):
    print(f'   {i}s remaining...')
    time.sleep(10)

print('\n🚀 Running autonomous loop now...')
result = run_autonomous_loop()
print('\n📊 Final result:')
print(json.dumps(result, indent=2, default=str))

## Cell 8 — Continuous Watch Loop (runs until stopped)

In [ ]:
# ⚠️  This cell runs continuously — Kernel > Interrupt to stop
POLL_INTERVAL = 60  # seconds between checks
loop_count = 0

print(f'🔄 Starting continuous watch loop (every {POLL_INTERVAL}s)')
print(f'   Kernel > Interrupt Kernel to stop')
print(f'   Results stream to Streamlit console in real time')
print(f'   HITL items appear immediately in the HITL Queue tab')

try:
    while True:
        loop_count += 1
        clear_output(wait=True)
        print(f'🔄 Continuous watch loop — iteration {loop_count}')
        print(f'   Poll interval: {POLL_INTERVAL}s | Ctrl+C or Kernel Interrupt to stop')
        result = run_autonomous_loop()
        print(f'\n💤 Next check in {POLL_INTERVAL}s...')
        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print(f'\n⛔ Watch loop stopped after {loop_count} iterations')

## Cell 9 — Manual: Inject Specific Faults for Testing

In [ ]:
# Run any of these individually to test different scenarios

# Scenario A — High latency on auth (should auto-remediate)
# requests.post('http://127.0.0.1:7002/fault/latency', json={'ms': 1200}, timeout=3)

# Scenario B — High error rate on checkout (HITL escalation)
# requests.post('http://127.0.0.1:7003/fault/errors', json={'pct': 0.5}, timeout=3)

# Scenario C — CPU spike on fraud (HITL — high risk)
# requests.post('http://127.0.0.1:7004/fault/cpu_spin', json={'seconds': 90}, timeout=3)

# Scenario D — Memory leak on payments
# requests.post('http://127.0.0.1:7001/fault/mem_leak', json={'mb_per_min': 100}, timeout=3)

# Clear all faults
# for port in [7001,7002,7003,7004]:
#     requests.post(f'http://127.0.0.1:{port}/fault/clear', json={}, timeout=3)

# Run one manual loop without waiting
result = run_autonomous_loop()
print(json.dumps({
    'incident_id':     result.get('incident_id'),
    'anomaly_count':   len(result.get('anomalies',[])),
    'root_cause':      result.get('rca',{}).get('root_cause'),
    'action':          result.get('rca',{}).get('action'),
    'auto_remediated': result.get('auto_remediated'),
    'duration_s':      result.get('loop_duration_s'),
}, indent=2))

## Cell 10 — View Audit Log & HITL Queue

In [ ]:
print('=== AUDIT LOG (last 5 events) ===')
if AUDIT_FILE.exists():
    lines = AUDIT_FILE.read_text().strip().split('\n')
    for line in lines[-5:]:
        evt = json.loads(line)
        print(f"  [{evt.get('timestamp','')}] {evt.get('event_type')} — "
              f"{evt.get('incident_id',evt.get('hitl_id',''))}")
else:
    print('  No audit log yet')

print('\n=== HITL QUEUE ===')
if HITL_FILE.exists():
    lines = HITL_FILE.read_text().strip().split('\n')
    for line in lines:
        if not line.strip():
            continue
        h = json.loads(line)
        status_icon = '⏳' if h['status']=='PENDING' else ('✅' if h['status']=='APPROVED' else '❌')
        print(f"  {status_icon} {h.get('hitl_id')} | {h.get('status')} | "
              f"{h.get('rca',{}).get('action','?')} → {h.get('rca',{}).get('action_target','?')}")
else:
    print('  No HITL items yet')

## Cell 11 — UC-3: Trend Forecasting Anomaly Detection (GPU)

Fits a **linear regression** on the last 15 minutes of each metric per service,
projects **5 minutes ahead**, and alerts if the projected value will breach threshold.
GPU (Qwen3-30B) generates a preemptive operator advisory and routes BREACH forecasts to HITL.

In [ ]:
# UC-3 — Trend Forecasting Anomaly Detection
# Detects services TRENDING toward breach BEFORE they cross threshold
# GPU: narrative generation + preemptive remediation recommendation

import numpy as np
import pandas as pd
import json, time, uuid
from pathlib import Path
from datetime import datetime, timezone

METRICS_CSV   = Path('/workspace/shared/minicluster/live_metrics.csv')
HITL_FILE     = Path('/workspace/shared/hitl_queue.jsonl')
AUDIT_FILE    = Path('/workspace/shared/audit_log.jsonl')
FORECAST_FILE = Path('/workspace/shared/uc3_forecasts.jsonl')
FORECAST_FILE.parent.mkdir(parents=True, exist_ok=True)

THRESHOLDS = {
    'cpu_utilization': 70.0,
    'latency_p95_ms':  500.0,
    'error_rate':      0.05,
    'mem_mb':          1800.0,
}

FORECAST_MINS     = 5
TREND_WINDOW_MINS = 15
WARN_PCT          = 0.80
SERVICES          = {'payments': 7001, 'auth': 7002, 'checkout': 7003, 'fraud': 7004}

def ts():
    return datetime.now(timezone.utc).isoformat()

def write_hitl(event):
    HITL_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(HITL_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def write_audit(event):
    AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(AUDIT_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def linear_forecast(series, forecast_steps=5):
    if len(series) < 3:
        return None, None
    x = np.arange(len(series))
    y = series.values
    A = np.column_stack([x, np.ones(len(x))])
    result = np.linalg.lstsq(A, y, rcond=None)
    m, b = result[0]
    projected = m * (len(series) + forecast_steps - 1) + b
    y_pred = m * x + b
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / (ss_tot + 1e-9)
    return float(projected), float(r2)

def run_uc3_forecast(verbose=True):
    print(f'\n{"="*60}')
    print(f'📈 UC-3 TREND FORECASTING — {datetime.now().strftime("%H:%M:%S")}')
    print(f'   Window: {TREND_WINDOW_MINS}min history → {FORECAST_MINS}min forecast')
    print(f'{"="*60}')

    df = pd.read_csv(METRICS_CSV, parse_dates=['timestamp'])
    cutoff = df['timestamp'].max() - pd.Timedelta(minutes=TREND_WINDOW_MINS)
    df = df[df['timestamp'] >= cutoff].copy()
    if df.empty:
        print('No data in window'); return []

    forecasts, alerts = [], []

    for svc in df['service'].unique():
        svc_df = df[df['service'] == svc].sort_values('timestamp')
        for metric, threshold in THRESHOLDS.items():
            if metric not in svc_df.columns:
                continue
            series = svc_df[metric].dropna()
            if len(series) < 3:
                continue
            current = float(series.iloc[-1])
            projected, r2 = linear_forecast(series, FORECAST_MINS)
            if projected is None:
                continue

            pct_of_thresh = projected / threshold
            trend_dir     = '↑' if projected > current else '↓'

            rec = {
                'service': svc, 'metric': metric,
                'current': round(current, 4), 'projected': round(projected, 4),
                'threshold': threshold, 'pct_of_thresh': round(pct_of_thresh, 3),
                'r2': round(r2, 3), 'trend': trend_dir,
                'delta': round(projected - current, 4),
                'forecast_mins': FORECAST_MINS, 'timestamp': ts(),
            }

            if projected > threshold:
                rec['severity'] = 'BREACH'
                alerts.append(rec)
                print(f'   🔴 {svc}.{metric} BREACH in {FORECAST_MINS}min: {current:.3f} → {projected:.3f} [R²={r2:.2f}]')
            elif pct_of_thresh >= WARN_PCT and trend_dir == '↑':
                rec['severity'] = 'APPROACHING'
                alerts.append(rec)
                print(f'   🟡 {svc}.{metric} APPROACHING: {current:.3f} → {projected:.3f} ({pct_of_thresh*100:.0f}%) [R²={r2:.2f}]')
            else:
                rec['severity'] = 'OK'
                if verbose:
                    print(f'   ✅ {svc}.{metric}: {current:.3f} → {projected:.3f} ({trend_dir}) R²={r2:.2f}')

            forecasts.append(rec)

    # Write forecast file for Streamlit tab
    with open(FORECAST_FILE, 'w') as f:
        for rec in forecasts:
            f.write(json.dumps(rec) + '\n')

    if not alerts:
        print('\n✅ No trend alerts — all services tracking safely')
        write_audit({'event_type': 'UC3_FORECAST_CLEAN', 'timestamp': ts()})
        return forecasts

    # GPU narrative
    narrative = 'No GPU narrative'
    print(f'\n🧠 Calling GPU for advisory ({len(alerts)} alerts)...')
    try:
        prompt = f"""You are a banking SRE. Services have metrics TRENDING toward threshold
breaches in the next {FORECAST_MINS} minutes (linear regression forecast).

Trending alerts: {json.dumps(alerts)}

Write a 3-sentence operator advisory:
1. Which services are most at risk and why
2. Likely cause based on the trend pattern
3. Preemptive action to take NOW before breach

Be specific and actionable. No JSON."""
        resp = llm.chat.completions.create(
            model='Qwen3-30B-A3B',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2, max_tokens=200,
            extra_body={'chat_template_kwargs': {'enable_thinking': False}}
        )
        narrative = resp.choices[0].message.content.strip()
        print(f'\n📋 GPU Advisory:\n{narrative}')
    except Exception as e:
        narrative = f'GPU unavailable: {e}'
        print(f'   ⚠️  {narrative}')

    # HITL for BREACH forecasts
    breach_alerts = [a for a in alerts if a['severity'] == 'BREACH']
    if breach_alerts:
        hitl_evt = {
            'hitl_id':     f'hitl-uc3-{uuid.uuid4().hex[:8]}',
            'incident_id': f'uc3-{uuid.uuid4().hex[:8]}',
            'timestamp':   ts(),
            'source':      'uc3_trend_forecaster',
            'service':     breach_alerts[0]['service'],
            'anomalies':   breach_alerts,
            'rca': {
                'root_cause':    f'Trend forecasting predicts breach in {FORECAST_MINS}min',
                'action':        'preemptive_clear',
                'action_target': breach_alerts[0]['service'],
                'confidence':    0.70,
                'risk_level':    'MEDIUM',
                'reasoning':     narrative,
            },
            'status': 'PENDING',
            'uc': 'UC-3',
        }
        write_hitl(hitl_evt)
        write_audit({**hitl_evt, 'event_type': 'UC3_HITL_CREATED'})
        print(f'\n🛑 HITL created: {hitl_evt["hitl_id"]} — check Streamlit HITL tab')

    write_audit({'event_type': 'UC3_FORECAST_ALERTS', 'timestamp': ts(),
                 'alert_count': len(alerts), 'narrative': narrative, 'alerts': alerts})
    return forecasts

# Run once
forecasts = run_uc3_forecast(verbose=True)
print(f'\n📊 Total forecasts: {len(forecasts)}')

In [ ]:
# UC-3 Continuous Forecast Loop — Kernel > Interrupt to stop
import time
loop_count = 0
print('🔄 UC-3 continuous trend forecasting every 60s — Interrupt to stop')
try:
    while True:
        loop_count += 1
        print(f'\n--- Iteration {loop_count} ---')
        run_uc3_forecast(verbose=False)
        print('💤 Next in 60s...')
        time.sleep(60)
except KeyboardInterrupt:
    print(f'\n⛔ Stopped after {loop_count} iterations')


In [ ]:
# UC-3 Test: inject a gradual escalating fault to create a visible trend
# Run this, then run the forecast cell — it should detect the upward trend
import requests, time

print('Injecting escalating latency on payments to create upward trend...')
for ms in [100, 200, 300, 400, 500]:
    r = requests.post('http://127.0.0.1:7001/fault/latency', json={'ms': ms}, timeout=3)
    print(f'  Latency set to {ms}ms: {r.json()}')
    time.sleep(65)  # wait for metric to be written

print('\nNow run the UC-3 forecast cell to detect the trend!')


## Cell 15 — UC-5: Silent Failure / Absence of Signal Detection (GPU)

Detects services that have **stopped reporting** or gone **suspiciously quiet** —
a flatline in metrics or a gap in the collector feed.
This is the failure mode threshold-based detection misses entirely.

**Detection logic:**
- Gap detection: last metric timestamp > N minutes ago → service silent
- Flatline detection: metric std dev ≈ 0 over last window → suspiciously static
- Zero-RPS detection: rps=0 but service is supposed to be live

**GPU role:** Qwen3-30B reasons about *why* silence is suspicious in a banking context
and recommends investigation priority.

In [25]:
# UC-5 — Silent Failure / Absence of Signal Detection
import numpy as np
import pandas as pd
import json, time, uuid
from pathlib import Path
from datetime import datetime, timezone

METRICS_CSV   = Path('/workspace/shared/minicluster/live_metrics.csv')
HITL_FILE     = Path('/workspace/shared/hitl_queue.jsonl')
AUDIT_FILE    = Path('/workspace/shared/audit_log.jsonl')
SILENCE_FILE  = Path('/workspace/shared/uc5_silence.jsonl')
SILENCE_FILE.parent.mkdir(parents=True, exist_ok=True)

EXPECTED_SERVICES  = ['payments', 'auth', 'checkout', 'fraud']
SILENCE_THRESH_MIN = 3      # minutes since last report = silent
FLATLINE_STD_THRESH = 0.001 # std dev below this = suspiciously flat
FLATLINE_WINDOW_MIN = 10    # window to check flatline
ZERO_RPS_WINDOW_MIN = 5     # window to check zero RPS
SERVICES = {'payments': 7001, 'auth': 7002, 'checkout': 7003, 'fraud': 7004}

def ts():
    return datetime.now(timezone.utc).isoformat()

def write_hitl(event):
    HITL_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(HITL_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def write_audit(event):
    AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(AUDIT_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def run_uc5_silence_detection(verbose=True):
    print(f'\n{"="*60}')
    print(f'🔇 UC-5 SILENCE DETECTION — {datetime.now().strftime("%H:%M:%S")}')
    print(f'   Gap>{SILENCE_THRESH_MIN}min | Flatline std<{FLATLINE_STD_THRESH} | Zero-RPS>{ZERO_RPS_WINDOW_MIN}min')
    print(f'{"="*60}')

    df = pd.read_csv(METRICS_CSV, parse_dates=['timestamp'])
    now = df['timestamp'].max()
    findings = []

    # ── Check 1: Gap detection — service not reported recently ────────────
    print('\n🔍 Check 1: Gap detection (last report time)')
    reported_services = df['service'].unique().tolist()

    for svc in EXPECTED_SERVICES:
        if svc not in reported_services:
            findings.append({
                'service': svc, 'type': 'MISSING_SERVICE',
                'severity': 'CRITICAL',
                'detail': f'Service {svc} has never reported — may be down or not started',
                'timestamp': ts(),
            })
            print(f'   🔴 {svc}: NEVER REPORTED — not in metrics at all')
            continue

        svc_df   = df[df['service'] == svc]
        last_ts  = svc_df['timestamp'].max()
        gap_mins = (now - last_ts).total_seconds() / 60

        if gap_mins > SILENCE_THRESH_MIN:
            findings.append({
                'service': svc, 'type': 'REPORTING_GAP',
                'severity': 'CRITICAL' if gap_mins > 10 else 'HIGH',
                'gap_mins': round(gap_mins, 2),
                'last_seen': str(last_ts),
                'detail': f'{svc} last reported {gap_mins:.1f}min ago — possible crash or network loss',
                'timestamp': ts(),
            })
            print(f'   🔴 {svc}: SILENT for {gap_mins:.1f}min (last={last_ts})')
        else:
            if verbose:
                print(f'   ✅ {svc}: reporting OK (last {gap_mins:.1f}min ago)')

    # ── Check 2: Flatline detection — metric stuck at constant value ───────
    print('\n🔍 Check 2: Flatline detection (std dev over last 10min)')
    window_df = df[df['timestamp'] >= now - pd.Timedelta(minutes=FLATLINE_WINDOW_MIN)]

    for svc in EXPECTED_SERVICES:
        svc_df = window_df[window_df['service'] == svc]
        if len(svc_df) < 3:
            continue
        for metric in ['latency_p95_ms', 'cpu_utilization', 'error_rate']:
            if metric not in svc_df.columns:
                continue
            series = svc_df[metric].dropna()
            if len(series) < 3:
                continue
            std  = float(series.std())
            mean = float(series.mean())
            if std < FLATLINE_STD_THRESH and mean > 0:
                findings.append({
                    'service': svc, 'type': 'FLATLINE',
                    'severity': 'HIGH',
                    'metric': metric,
                    'std': round(std, 6),
                    'mean': round(mean, 4),
                    'detail': f'{svc}.{metric} flatlined at {mean:.4f} (std={std:.6f}) — possible stuck metric or dead collector',
                    'timestamp': ts(),
                })
                print(f'   🟡 {svc}.{metric}: FLATLINE mean={mean:.4f} std={std:.6f}')
            else:
                if verbose:
                    print(f'   ✅ {svc}.{metric}: varying OK (std={std:.4f})')

    # ── Check 3: Zero RPS — service alive but serving no traffic ──────────
    print('\n🔍 Check 3: Zero-RPS detection (last 5min)')
    rps_window = df[df['timestamp'] >= now - pd.Timedelta(minutes=ZERO_RPS_WINDOW_MIN)]

    for svc in EXPECTED_SERVICES:
        svc_df = rps_window[rps_window['service'] == svc]
        if 'rps' not in svc_df.columns or len(svc_df) < 2:
            continue
        avg_rps = float(svc_df['rps'].mean())
        if avg_rps == 0.0:
            findings.append({
                'service': svc, 'type': 'ZERO_RPS',
                'severity': 'HIGH',
                'avg_rps': 0.0,
                'window_mins': ZERO_RPS_WINDOW_MIN,
                'detail': f'{svc} serving 0 RPS for {ZERO_RPS_WINDOW_MIN}min — traffic dropped entirely',
                'timestamp': ts(),
            })
            print(f'   🟡 {svc}: ZERO RPS for last {ZERO_RPS_WINDOW_MIN}min')
        else:
            if verbose:
                print(f'   ✅ {svc}: rps={avg_rps:.2f}')

    # Write findings file for Streamlit
    with open(SILENCE_FILE, 'w') as f:
        for rec in findings:
            f.write(json.dumps(rec) + '\n')

    print(f'\n📊 Findings: {len(findings)} ({sum(1 for f in findings if f["severity"]=="CRITICAL")} CRITICAL)')

    if not findings:
        print('✅ No silence anomalies detected')
        write_audit({'event_type': 'UC5_CLEAN', 'timestamp': ts()})
        return findings

    # GPU reasoning
    narrative = 'No GPU narrative'
    print(f'\n🧠 Calling GPU for silence reasoning...')
    try:
        prompt = f"""You are a senior banking SRE investigating silent failures.
The following anomalies were detected — services that stopped reporting, flatlined, or dropped to zero traffic:

{json.dumps(findings, indent=2)}

In a banking microservices context (payments, auth, checkout, fraud), write a 3-sentence analysis:
1. Which finding is most operationally dangerous and why
2. Most likely root cause (network partition, OOM crash, deadlock, upstream dependency failure)
3. First two investigation steps an SRE should take RIGHT NOW

No JSON. Be specific and urgent."""

        resp = llm.chat.completions.create(
            model='Qwen3-30B-A3B',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2, max_tokens=200,
            extra_body={'chat_template_kwargs': {'enable_thinking': False}}
        )
        narrative = resp.choices[0].message.content.strip()
        print(f'\n📋 GPU Analysis:\n{narrative}')
    except Exception as e:
        narrative = f'GPU unavailable: {e}'

    # HITL for CRITICAL findings
    critical = [f for f in findings if f['severity'] == 'CRITICAL']
    if critical:
        hitl_evt = {
            'hitl_id':     f'hitl-uc5-{uuid.uuid4().hex[:8]}',
            'incident_id': f'uc5-{uuid.uuid4().hex[:8]}',
            'timestamp':   ts(),
            'source':      'uc5_silence_detector',
            'service':     critical[0]['service'],
            'anomalies':   critical,
            'rca': {
                'root_cause':    'Service silence / absence of signal detected',
                'action':        'investigate_and_restart',
                'action_target': critical[0]['service'],
                'confidence':    0.65,
                'risk_level':    'HIGH',
                'reasoning':     narrative,
            },
            'status': 'PENDING',
            'uc': 'UC-5',
        }
        write_hitl(hitl_evt)
        write_audit({**hitl_evt, 'event_type': 'UC5_HITL_CREATED'})
        print(f'\n🛑 HITL created: {hitl_evt["hitl_id"]}')

    write_audit({'event_type': 'UC5_FINDINGS', 'timestamp': ts(),
                 'finding_count': len(findings), 'narrative': narrative, 'findings': findings})
    return findings

# Run once
findings = run_uc5_silence_detection(verbose=True)



🔇 UC-5 SILENCE DETECTION — 13:02:38
   Gap>3min | Flatline std<0.001 | Zero-RPS>5min

🔍 Check 1: Gap detection (last report time)
   ✅ payments: reporting OK (last 0.0min ago)
   ✅ auth: reporting OK (last 0.0min ago)
   ✅ checkout: reporting OK (last 0.0min ago)
   ✅ fraud: reporting OK (last 0.0min ago)

🔍 Check 2: Flatline detection (std dev over last 10min)
   ✅ payments.latency_p95_ms: varying OK (std=0.0063)
   ✅ payments.cpu_utilization: varying OK (std=0.0539)
   ✅ payments.error_rate: varying OK (std=0.0000)
   🟡 auth.latency_p95_ms: FLATLINE mean=1.2600 std=0.000000
   ✅ auth.cpu_utilization: varying OK (std=0.0522)
   ✅ auth.error_rate: varying OK (std=0.0664)
   ✅ checkout.latency_p95_ms: varying OK (std=0.0045)
   ✅ checkout.cpu_utilization: varying OK (std=0.0820)
   ✅ checkout.error_rate: varying OK (std=0.0970)
   ✅ fraud.latency_p95_ms: varying OK (std=0.0030)
   ✅ fraud.cpu_utilization: varying OK (std=0.0751)
   ✅ fraud.error_rate: varying OK (std=0.0000)

🔍 Check 3

In [ ]:
# UC-5 Continuous Silence Watch — Kernel > Interrupt to stop
import time
loop_count = 0
print('🔇 UC-5 continuous silence detection every 60s — Interrupt to stop')
try:
    while True:
        loop_count += 1
        findings = run_uc5_silence_detection(verbose=False)
        sev_counts = {s: sum(1 for f in findings if f['severity']==s) for s in ['CRITICAL','HIGH']}
        print(f'Iter {loop_count}: {len(findings)} findings — CRITICAL:{sev_counts["CRITICAL"]} HIGH:{sev_counts["HIGH"]}')
        print('💤 Next in 60s...')
        time.sleep(60)
except KeyboardInterrupt:
    print(f'\n⛔ Stopped after {loop_count} iterations')


In [ ]:
# UC-5 Test: simulate silence by stopping the collector
# WARNING: this stops metric collection — restart supervisord after testing
import subprocess, time

print('Stopping collector to simulate service silence...')
subprocess.run(['supervisorctl', '-c', '/workspace/shared/minicluster/supervisord.conf',
                'stop', 'collector'], capture_output=True)
print('Collector stopped. Waiting 4 minutes for gap to appear in metrics...')
time.sleep(240)
print('Now run the UC-5 detection cell — should see REPORTING_GAP findings')
print('\nTo restore: supervisorctl restart collector')
